# Kernel PCA

## The Limitation of Linear PCA

Standard PCA finds directions of maximum variance using **linear projections**. It works well when the structure of the data can be captured by straight lines or hyperplanes.

But what if the class boundaries are curved? What if the data is arranged in rings, spirals, or other non-linear patterns? Linear PCA would project everything onto a straight axis and destroy the class structure.

---

## The Kernel Trick: Linear PCA in a Higher-Dimensional Space

Kernel PCA extends standard PCA using the **kernel trick** — the same idea as in Kernel SVM.

Instead of finding linear directions in the original feature space, Kernel PCA implicitly maps the data into a **higher-dimensional space** where non-linear patterns become linear, then applies PCA there.

```
Original space (non-linear):          Kernel space (mapped):

  Class A: ooo                           Class A:  ooo
  Class B: x o x o (mixed)    →          Class B:           xxx
  Not linearly separable                 Linearly separable!
```

**The trick:** You never explicitly compute the high-dimensional coordinates. You only compute dot products between data points — and dot products in the mapped space can be computed using a **kernel function** $K(x_i, x_j) = \phi(x_i) \cdot \phi(x_j)$ without ever knowing $\phi$.

---

## The RBF Kernel

The Radial Basis Function (RBF) kernel is the most common choice:

$$K(x_i, x_j) = \exp\left(-\gamma \|x_i - x_j\|^2\right)$$

It measures similarity based on **distance**: nearby points have kernel value near 1, far-apart points have kernel value near 0. The RBF kernel implicitly maps to an infinite-dimensional space — it can capture arbitrarily complex non-linear patterns.

---

## PCA vs Kernel PCA vs LDA: Which to Choose?

| Situation | Choose |
|-----------|--------|
| Data has linear structure, no labels | **PCA** |
| Data has linear structure, class labels available | **LDA** |
| Data has non-linear structure | **Kernel PCA** |
| Need interpretable components | **PCA or LDA** (Kernel PCA components have no interpretable meaning) |
| Large dataset (>50K samples) | **PCA** (Kernel PCA is O(n²) in memory) |

---

## What We Will Build

Same Wine dataset and Logistic Regression setup as the PCA and LDA notebooks. This lets us compare all three dimensionality reduction methods on the same data:
- PCA: ~97% accuracy
- LDA: ~100% accuracy
- Kernel PCA: how does it compare?

## Step 1: Import Libraries

Same three standard libraries. `KernelPCA` is available from `sklearn.decomposition` alongside standard PCA.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Step 2: Load the Dataset

Same Wine dataset as the PCA and LDA notebooks: 178 samples, 13 features, 3 classes.

Using the same dataset across all three dimensionality reduction notebooks lets you make a direct apples-to-apples comparison of the methods — the only variable changing is the reduction technique.

In [ ]:
dataset = pd.read_csv('Wine.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

## Step 3: Train/Test Split

80/20 split, same as before. The split comes before Kernel PCA fitting — we learn the kernel mapping from training data only, then apply it to the test set.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

## Step 4: Feature Scaling

Kernel PCA depends critically on feature scaling. The RBF kernel computes **Euclidean distances** between data points:

$$K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$$

If features have different scales, the distance calculation is dominated by the large-scale feature. A difference of 1,000 in salary will swamp a difference of 0.5 in alcohol content, even if the alcohol difference is more discriminative.

After StandardScaler, all features are on the same scale and the kernel computes meaningful, balanced distances.

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Step 5: Apply Kernel PCA

```python
kpca = KernelPCA(n_components=2, kernel='rbf')
X_train = kpca.fit_transform(X_train)
X_test  = kpca.transform(X_test)
```

**`kernel='rbf'`** — uses the Radial Basis Function kernel. Other options include `'poly'` (polynomial), `'sigmoid'`, and `'cosine'`.

**Key difference from standard PCA:** The components produced by Kernel PCA are in the **mapped feature space**, not the original space. They are not linear combinations of the original 13 wine features — they are non-linear combinations that may capture curved cluster boundaries.

**Key difference from LDA:** Kernel PCA is **unsupervised** — it does not use `y_train`. It finds the principal components of the kernel matrix, which captures non-linear variance structure without requiring class labels.

**Computational note:** Kernel PCA requires computing an $n \times n$ kernel matrix where $n$ is the number of training samples. For 8,000 training samples, this is a 64 million entry matrix. For datasets with 100K+ samples, standard PCA is much more practical.

In [ ]:
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(n_components = 2, kernel = 'rbf')
X_train = kpca.fit_transform(X_train)
X_test = kpca.transform(X_test)

## Step 6: Train Logistic Regression on Kernel PCA Features

The two Kernel PCA components capture non-linear structure in the wine dataset. If the wine classes have curved boundaries in the original 13D space, those boundaries should appear more linear in the Kernel PCA projection — making logistic regression more effective.

In [ ]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(random_state = 0)
classifier.fit(X_train, y_train)

## Step 7: Evaluate — Confusion Matrix

Compare the result here against PCA (~97%) and LDA (~100%).

**Interpreting the result:**

If Kernel PCA matches LDA (100%), it means the wine dataset does have non-linear structure that the RBF kernel captured effectively.

If it matches standard PCA (~97%), the dataset's class structure is approximately linear — the kernel mapping did not help.

**A broader lesson:** More complex does not always mean better. Kernel PCA is a more powerful but more expensive method than PCA. If the data is approximately linear, standard PCA will work just as well at a fraction of the computational cost. Always start simple.

**When Kernel PCA shines:** Data arranged in rings (inner class vs outer class), crescents, spirals, or any other shape where linear projections destroy class structure. The classic demonstration is the Swiss roll or concentric circles dataset.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

## Step 8: Visualise the Training Set

The axes are now **non-linear principal components** — they have no interpretable meaning as combinations of the original wine features.

Look for: how cleanly do the three wine classes separate in this 2D kernel space? Compare visually to the PCA and LDA plots from the previous notebooks.

In [ ]:
from matplotlib.colors import ListedColormap
X_set, y_set = X_train, y_train
X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 1, stop = X_set[:, 0].max() + 1, step = 0.01),
                     np.arange(start = X_set[:, 1].min() - 1, stop = X_set[:, 1].max() + 1, step = 0.01))
plt.contourf(X1, X2, classifier.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
             alpha = 0.75, cmap = ListedColormap(('red', 'green', 'blue')))
plt.xlim(X1.min(), X1.max())
plt.ylim(X2.min(), X2.max())
for i, j in enumerate(np.unique(y_set)):
    plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                c = ListedColormap(('red', 'green', 'blue'))(i), label = j)
plt.title('Logistic Regression (Training set)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.show()

## Step 9: Visualise the Test Set

The test set result is the honest comparison.

**Three-way comparison summary:**

| Method | Type | Uses labels | Wine accuracy | Best for |
|--------|------|------------|---------------|----------|
| PCA | Linear, unsupervised | No | ~97% | Exploration, noise reduction |
| LDA | Linear, supervised | Yes | ~100% | Maximising class separability |
| Kernel PCA | Non-linear, unsupervised | No | ~100% | Non-linear structure |

For the Wine dataset — which has a relatively clean, near-linear structure — all three methods perform well. The differences become dramatic on datasets with genuinely non-linear class boundaries.

In [ ]:
from matplotlib.colors import ListedColormap
X_set, y_set = X_test, y_test
X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 1, stop = X_set[:, 0].max() + 1, step = 0.01),
                     np.arange(start = X_set[:, 1].min() - 1, stop = X_set[:, 1].max() + 1, step = 0.01))
plt.contourf(X1, X2, classifier.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
             alpha = 0.75, cmap = ListedColormap(('red', 'green', 'blue')))
plt.xlim(X1.min(), X1.max())
plt.ylim(X2.min(), X2.max())
for i, j in enumerate(np.unique(y_set)):
    plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                c = ListedColormap(('red', 'green', 'blue'))(i), label = j)
plt.title('Logistic Regression (Test set)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.show()